In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/anuragabcr@gmail.com/learning_spark/fmcg_atlikon_project/03_utilities

In [0]:
print(gold_schema, silver_schema, bronze_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://fmcg-de-project/{data_source}/*.csv"

print(base_path)

In [0]:
df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(base_path)\
    .withColumn("read_timestamp", F.current_timestamp())\
    .select("*", "_metadata.file_name", "_metadata.file_size")

display(df.limit(10))


In [0]:
df.write.format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
bronze_df = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
display(bronze_df.limit(10))

In [0]:
# duplicate_cid = bronze_df.groupBy("customer_id").count().filter("count > 1")
duplicate_cid = bronze_df.groupBy("customer_id").count().filter(F.col("count") > 1)
display(duplicate_cid)

In [0]:
print("Rows before duplicate drops", bronze_df.count())
silver_df = bronze_df.dropDuplicates(["customer_id"])
print("Rows after duplicate drops", bronze_df.count())

In [0]:
display(
    silver_df.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
silver_df = silver_df.withColumn("customer_name", F.trim(F.col("customer_name")))

In [0]:
display(silver_df.select("city").distinct())

In [0]:
city_mapping = {
 "Bengalore": "Bengaluru",
 "Bengaluruu": "Bengaluru",
 "Hyderabadd": "Hyderabad",
 "Hyderbad": "Hyderabad",
 "NewDelhi": "New Delhi",
 "NewDelhee": "New Delhi",
 "NewDheli": "New Delhi",
}

allowed_cities = ["Bengaluru", "Hyderabad", "New Delhi"]

silver_df = silver_df.replace(city_mapping, subset=["city"])\
            .withColumn("city",
                        F.when(F.col("city").isNull(), None)
                        .when(F.col("city").isin(allowed_cities), F.col("city"))
                        .otherwise(None)
                        )
            
display(silver_df.select("city").distinct())

In [0]:
display(silver_df.select("customer_name").distinct())

In [0]:
silver_df = silver_df.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
    .otherwise(F.initcap(F.col("customer_name")))
)
display(silver_df.select("customer_name").distinct())

In [0]:
display(silver_df.filter(F.col("city").isNull()))

In [0]:
null_city_customers = ["Sprintx Nutrition", "Zenathlete Foods", "Primefuel Nutrition", "Recovery Lane"]
display(silver_df.filter(F.col("customer_name").isin(null_city_customers)))

In [0]:
customer_city_fix = {
    789403: "New Delhi",
    789420: "Bengaluru",
    789521: "Hyderabad",
    789603: "Hyderabad"
}
df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
    )

In [0]:
silver_df =  silver_df.join(df_fix, on="customer_id", how="left")\
                .withColumn("city", F.coalesce(F.col("city"), F.col("fixed_city")))\
                .drop("fixed_city")

display(silver_df.filter(F.col("city").isNull()))

In [0]:
silver_df = silver_df.withColumn("customer_id", F.col("customer_id").cast("string"))
display(silver_df.limit(5))

In [0]:
silver_df = silver_df\
            .withColumn("customer_code", F.col("customer_id"))\
            .withColumn("customer", F.concat(F.col("customer_name"), F.lit("-"), F.col("city")))\
            .withColumn("market", F.lit("India"))\
            .withColumn("platform", F.lit("Sports India"))\
            .withColumn("channel", F.lit("Acquisition"))\
            .drop("customer_id", "customer_name", "city")

display(silver_df.limit(5))

In [0]:
silver_df.write.format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
silver_df = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")

gold_df = silver_df.select(["customer_code", "customer", "market", "platform", "channel"])
display(gold_df.limit(5))

In [0]:
gold_df.write.format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_customers").select("*")

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()